In [ ]:
import requests
import pandas as pd
from datetime import datetime
import json

def fetch_stock_data(url):
    """
    Fetch stock data from the API URL
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        return data
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return None

def convert_to_csv(data, filename='stock_data.csv'):
    """
    Convert the fetched data to CSV format with proper timestamp conversion
    """
    if not data:
        print("No data to convert")
        return
    
    # Extract data arrays
    timestamps = data.get('t', [])
    opens = data.get('o', [])
    highs = data.get('h', [])
    lows = data.get('l', [])
    closes = data.get('c', [])
    volumes = data.get('v', [])
    
    # Check if all arrays have the same length
    lengths = [len(timestamps), len(opens), len(highs), len(lows), len(closes), len(volumes)]
    if len(set(lengths)) > 1:
        print(f"Warning: Data arrays have different lengths: {lengths}")
    
    # Create a list to store the converted data
    csv_data = []
    
    for i in range(len(timestamps)):
        # Convert Unix timestamp to datetime
        dt = datetime.fromtimestamp(timestamps[i])
        
        row = {
            'Date': dt.strftime('%Y-%m-%d'),
            'Time': dt.strftime('%H:%M:%S'),
            'DateTime': dt.strftime('%Y-%m-%d %H:%M:%S'),
            'Open': opens[i] if i < len(opens) else None,
            'High': highs[i] if i < len(highs) else None,
            'Low': lows[i] if i < len(lows) else None,
            'Close': closes[i] if i < len(closes) else None,
            'Volume': volumes[i] if i < len(volumes) else None,
            'Unix_Timestamp': timestamps[i]
        }
        csv_data.append(row)
    
    # Create DataFrame
    df = pd.DataFrame(csv_data)
    
    # Save to CSV
    df.to_csv(filename, index=False)
    print(f"Data saved to {filename}")
    print(f"Total records: {len(df)}")
    
    return df

# Main execution
if __name__ == "__main__":
    # The API URL
    url = "https://nepsealpha.com/trading/1/history?fsk=a7Wiju0jpMOwX09n&symbol=NEPSE&resolution=1D&pass=ok"
    
    print("Fetching stock data...")
    data = fetch_stock_data(url)
    
    if data:
        print("Data fetched successfully!")
        print(f"Available keys: {list(data.keys())}")
        
        # Show sample of raw data
        if 't' in data:
            print(f"Number of timestamps: {len(data['t'])}")
            if data['t']:
                sample_timestamp = data['t'][0]
                sample_date = datetime.fromtimestamp(sample_timestamp)
                print(f"First timestamp: {sample_timestamp} -> {sample_date}")
        
        # Convert to CSV
        df = convert_to_csv(data, 'NEPSE_stock_data.csv')
        
        # Display first few rows
        if df is not None and not df.empty:
            print("\nFirst 5 rows of the data:")
            print(df.head())
            
            print("\nData summary:")
            print(df.describe())
    else:
        print("Failed to fetch data")

# Alternative: If you want to customize the URL
def fetch_custom_stock_data(symbol, fsk_token, resolution='1D'):
    """
    Fetch stock data for a custom symbol
    """
    url = f"https://nepsealpha.com/trading/1/history?fsk={fsk_token}&symbol={symbol}&resolution={resolution}&pass=ok"
    return fetch_stock_data(url)

# Example usage for different stocks:
# data = fetch_custom_stock_data('NABIL', 'BQHo4mdNw6B5Zk0b', '1D')
# df = convert_to_csv(data, 'NABIL_stock_data.csv')

In [1]:
import pandas as pd

# Example DataFrame (like yours)
df = pd.read_csv('result.csv')

# Convert UNIX timestamps (in seconds) → datetime
df['t'] = pd.to_datetime(df['t'], unit='s')

print(df)


        s        o        h        l        c             v          t
0      ok   173.26   173.26   173.26   173.26  0.000000e+00 1997-07-20
1     NaN   173.25   173.25   173.25   173.25  0.000000e+00 1997-07-21
2     NaN   173.03   173.03   173.03   173.03  0.000000e+00 1997-07-22
3     NaN   173.03   173.03   173.03   173.03  0.000000e+00 1997-07-23
4     NaN   172.66   172.66   172.66   172.66  0.000000e+00 1997-07-24
...   ...      ...      ...      ...      ...           ...        ...
6375  NaN  2610.28  2610.31  2535.30  2559.59  5.762701e+09 2025-10-08
6376  NaN  2585.73  2677.09  2575.51  2582.18  5.273221e+09 2025-10-09
6377  NaN  2583.97  2584.24  2493.16  2503.35  6.707245e+09 2025-10-12
6378  NaN  2521.01  2547.81  2478.39  2547.60  4.975409e+09 2025-10-13
6379  NaN  2547.77  2553.57  2523.58  2528.16  3.366198e+09 2025-10-14

[6380 rows x 7 columns]


In [3]:
df.rename(columns={'t': 'date'}, inplace=True)

# Save to a new CSV file
df.to_csv('converted_stock_data.csv', index=False)

In [ ]:
!pip install ta

In [5]:
df.set_index("date")

,s,o,h,l,c,v
date,,,,,,
1997-07-20,ok,173.26,173.26,173.26,173.26,0.000000e+00
1997-07-21,NaN,173.25,173.25,173.25,173.25,0.000000e+00
1997-07-22,NaN,173.03,173.03,173.03,173.03,0.000000e+00
1997-07-23,NaN,173.03,173.03,173.03,173.03,0.000000e+00
1997-07-24,NaN,172.66,172.66,172.66,172.66,0.000000e+00
...,...,...,...,...,...,...
2025-10-08,NaN,2610.28,2610.31,2535.30,2559.59,5.762701e+09
2025-10-09,NaN,2585.73,2677.09,2575.51,2582.18,5.273221e+09
2025-10-12,NaN,2583.97,2584.24,2493.16,2503.35,6.707245e+09


In [7]:
import ta

# Add all indicators at once
# df['RSI'] = ta.momentum.RSIIndicator(close=df['c'], window=14).rsi()
# df['EMA_20'] = ta.trend.EMAIndicator(close=df['c'], window=20).ema_indicator()
# df['SMA_50'] = ta.trend.SMAIndicator(close=df['c'], window=50).sma_indicator()
# df['MACD'] = ta.trend.MACD(close=df['c']).macd()
# df['MACD_signal'] = ta.trend.MACD(close=df['c']).macd_signal()
import pandas as pd
import ta

# Suppose df has columns: 'o', 'h', 'l', 'c', 'v'
# (You can rename them if your columns differ)
df['psar'] = ta.trend.PSARIndicator(
    high=df['h'], low=df['l'], close=df['c']
).psar()

# Add +DI, -DI, and ADX
adx = ta.trend.ADXIndicator(
    high=df['h'], low=df['l'], close=df['c'], window=5
)
df['PosDI'] = adx.adx_pos()
df['NegDI'] = adx.adx_neg()
df['ADX'] = adx.adx()

# print(df[['c', 'psar', '+DI', '-DI', 'ADX']])


In [9]:
df

,s,o,h,l,c,v,date,psar,PosDI,NegDI,ADX
0,ok,173.26,173.26,173.26,173.26,0.000000e+00,1997-07-20,173.260000,0.000000,0.000000,0.000000
1,NaN,173.25,173.25,173.25,173.25,0.000000e+00,1997-07-21,173.250000,0.000000,0.000000,0.000000
2,NaN,173.03,173.03,173.03,173.03,0.000000e+00,1997-07-22,173.260000,0.000000,0.000000,0.000000
3,NaN,173.03,173.03,173.03,173.03,0.000000e+00,1997-07-23,173.255400,0.000000,0.000000,0.000000
4,NaN,172.66,172.66,172.66,172.66,0.000000e+00,1997-07-24,173.250892,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
6375,NaN,2610.28,2610.31,2535.30,2559.59,5.762701e+09,2025-10-08,2490.615788,9.508549,53.429926,52.298409
6376,NaN,2585.73,2677.09,2575.51,2582.18,5.273221e+09,2025-10-09,2494.237072,25.134454,35.788463,45.336265
6377,NaN,2583.97,2584.24,2493.16,2503.35,6.707245e+09,2025-10-12,2677.090000,19.042366,49.028844,45.079338
6378,NaN,2521.01,2547.81,2478.39,2547.60,4.975409e+09,2025-10-13,2677.090000,15.469983,43.822413,45.627068


In [11]:
df.to_csv('NEPSE_ADX5_data.csv', index=False)